# 04 — Evaluation, Benchmark Comparison & Model Registry

**Team Update 4.1 step 3** (evaluate the model and compare it against the
simple benchmark) **+ the Model Store** (group, package, model card).

| Step | Verify (📸) |
|---|---|
| Champion selection (val macro F1 + High-Risk recall floor, benchmark gate) | selection table |
| Test-set confusion matrix + fairness slices + feature importance | plots below |
| Registry: model group, package, model card | `describe_*` outputs |

Champion rule (design doc): best **validation** macro F1 subject to
High-Risk recall >= floor, then confirmed once on the untouched test split,
and required to beat the benchmark's test macro F1.

## 1. Setup

In [ ]:
%pip install -q "sagemaker<3" xgboost matplotlib

import json
import subprocess
import tarfile

import boto3
import pandas as pd
import sagemaker

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
bucket = sagemaker.Session(boto_session=boto_session).default_bucket()
prefix = "aai-540-g1"

try:
    role = sagemaker.get_execution_role()  # Studio / notebook instance
except Exception:
    # local run (personal account): dedicated project role if present,
    # otherwise the newest SageMaker execution role in the account
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        if not cands:
            raise RuntimeError("no SageMaker execution role found - run scripts/ensure_execution_role.py")
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"
print(f"role={role}\nbucket={bucket}")

## 2. Load results + pick the champion

In [ ]:
s3.download_file(bucket, f"{prefix}/artifacts/evaluation/training_comparison.csv",
                 "training_comparison.csv")
s3.download_file(bucket, f"{prefix}/artifacts/evaluation/benchmark_metrics.json",
                 "benchmark_metrics.json")
comparison = pd.read_csv("training_comparison.csv")
benchmark = json.load(open("benchmark_metrics.json"))
benchmark_test_f1 = benchmark["b2_visits_tercile_rule"]["macro_f1"]
print(f"benchmark (B2 visits-tercile) test macro F1: {benchmark_test_f1:.4f}")

RECALL_FLOOR = 0.70
qualified = comparison[comparison["val_high_recall"] >= RECALL_FLOOR]
champion = qualified.iloc[0]  # comparison is sorted by val macro F1 desc
print(f"champion: {champion['run']} "
      f"(val F1 {champion['val_macro_f1']:.4f}, "
      f"test F1 {champion['test_macro_f1']:.4f}, "
      f"test High-recall {champion['test_high_recall']:.4f})")
assert champion["test_macro_f1"] > benchmark_test_f1, \
    "champion does not beat the benchmark - revisit training before registry"
comparison

## 3. Test-set deep dive: confusion matrix, slices, feature importance

Fetches the champion's per-member test predictions and its own metrics JSON
out of the model tarball (`train.py` writes them into the artifact), and
merges the audit columns (gender / race / age_band) from the training table.
`race` was excluded from features - it is used here purely as a fairness
slice, per the design doc.

In [ ]:
RUN = champion["run"]
artifact_key = f"{prefix}/artifacts/training/{RUN}/{RUN}/output/model.tar.gz"
s3.download_file(bucket, artifact_key, "champion.tar.gz")

with tarfile.open("champion.tar.gz") as tar:
    names = tar.getnames()
    preds = pd.read_csv(tar.extractfile(
        next(m for m in names if m.endswith("_test_predictions.csv"))))
    champ_metrics = json.loads(tar.extractfile(
        next(m for m in names if m.endswith("_metrics.json"))).read())
    model_member = next(m for m in names
                        if m.endswith(".bin") or m.endswith(".joblib"))
    ext = ".bin" if model_member.endswith(".bin") else ".joblib"
    with open(f"champion_model{ext}", "wb") as fh:
        fh.write(tar.extractfile(model_member).read())

s3.download_file(bucket, f"{prefix}/processed/training/training_table.csv",
                 "training_table.csv")
table = pd.read_csv("training_table.csv", dtype={"person_id": int})
preds = preds.merge(table[["person_id", "gender", "race", "age_band"]],
                    on="person_id", how="left")
RISK_CLASSES = ["Low", "Medium", "High"]
print(f"{len(preds):,} test predictions; hyperparameters:",
      champ_metrics["hyperparameters"])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    preds["y_true"], preds["y_pred"], labels=RISK_CLASSES,
    cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"Champion {RUN} - test confusion matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
# fairness slices: per-class recall by audit dimension (cells with n >= 30)
rows = []
for dim in ["gender", "race", "age_band"]:
    for value, grp in preds.groupby(dim):
        for cls in RISK_CLASSES:
            mask = grp["y_true"] == cls
            if mask.sum() >= 30:
                rec = (grp.loc[mask, "y_pred"] == cls).mean()
                rows.append({"slice": dim, "value": str(value), "class": cls,
                             "recall": round(float(rec), 3), "n": int(mask.sum())})
slices = pd.DataFrame(rows)
slices[slices["class"] == "High"].sort_values(["slice", "recall"])

In [ ]:
# champion booster feature importance (history-window signals dominate;
# the label comes from FUTURE utilization, so top slots should be chronic
# flags + visit counts, never a leakage column)
import sys
sys.path.insert(0, "src")
from model_common import encode_features, load_table
from xgboost import XGBClassifier, plot_importance

X_all, feature_names, _ = encode_features(load_table("training_table.csv"))
m = XGBClassifier()
m.load_model("champion_model.bin")
m.get_booster().feature_names = list(feature_names)

fig, ax = plt.subplots(figsize=(7, 6))
plot_importance(m.get_booster(), max_num_features=15, ax=ax)
ax.set_title(f"Champion {RUN} - top feature importances")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120)
plt.show()

## 4. Repackage a clean champion artifact

The training tarball also carries metrics/prediction files (they share
`SM_MODEL_DIR`); a serving-ready tarball should contain only the booster,
named `model.bin` as the XGBoost container expects.

In [ ]:
import subprocess

subprocess.run(
    ["tar", "-czf", "model.tar.gz", "--transform",
     f"s/champion_model{ext}/model.bin/", f"champion_model{ext}"],
    check=True)
s3.upload_file("model.tar.gz", bucket,
               f"{prefix}/artifacts/models/champion/model.tar.gz")
MODEL_DATA_URL = f"s3://{bucket}/{prefix}/artifacts/models/champion/model.tar.gz"
print(f"clean artifact: {MODEL_DATA_URL}")

## 5. Model Registry — group, package, card

boto3 calls reuse the patterns debugged in Assignment 4.1 (`XGBOOST` casing,
`describe_model_package(ModelPackageName=...)`, `json.dumps` card content,
schema-compliant sections).

In [ ]:
GROUP = "member-utilization-risk"
DESCRIPTION = (
    "Versioned lineage of multiclass classifiers predicting member utilization "
    "risk (Low/Medium/High) from 2008 history-window features on CMS DE-SynPUF "
    "(OMOP). A new package is added whenever algorithm, features, labels, or "
    "hyperparameters change."
)
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=GROUP, ModelPackageGroupDescription=DESCRIPTION)
    print(f"created group {GROUP}")
except sm_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"group {GROUP} already exists")
    else:
        raise

In [ ]:
XGB_IMAGE = sagemaker.image_uris.retrieve("xgboost", region, version="1.7-1")

pkg = sm_client.create_model_package(
    ModelPackageGroupName=GROUP,
    ModelPackageDescription=(
        f"Champion run {RUN}: XGBoost 1.7-1 multiclass (multi:softprob, "
        f"num_class=3), temporal split 40/10/10/40. Validation macro F1 "
        f"{champion['val_macro_f1']:.4f}; test macro F1 "
        f"{champion['test_macro_f1']:.4f}, High-Risk recall "
        f"{champion['test_high_recall']:.4f}. Beats benchmark "
        f"({benchmark_test_f1:.4f})."),
    InferenceSpecification={
        "Containers": [{
            "Image": XGB_IMAGE,
            "ModelDataUrl": MODEL_DATA_URL,
            "Framework": "XGBOOST",
            "FrameworkVersion": "1.7-1",
        }],
        "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus="PendingManualApproval",
)
MODEL_PACKAGE_ARN = pkg["ModelPackageArn"]
print(MODEL_PACKAGE_ARN)

In [ ]:
# Model card content follows the enforced JSON schema (Assignment 4.1 fix):
# six top-level sections only; hyperparameters inside training_job_details;
# extras in additional_information.custom_details.
model_card = {
    "model_overview": {
        "model_description": (
            "XGBoost gradient-boosted-trees multiclass classifier assigning CMS "
            "DE-SynPUF members a Low / Medium / High utilization-risk tier from "
            "2008 history-window features (demographics, utilization counts, "
            "chronic-condition flags, pharmacy intensity, breadth). Champion of "
            f"run {RUN}; beats both benchmark heuristics on the held-out test split."
        ),
        "model_creator": "Sourangshu Pal (AAI-540 Group 1)",
        "model_owner": "Sourangshu Pal (AAI-540 Group 1)",
        "model_artifact": [MODEL_DATA_URL],
        "algorithm_type": "XGBoost (gradient-boosted decision trees)",
        "problem_type": "Supervised multiclass classification (3 risk tiers)",
    },
    "intended_uses": {
        "purpose_of_model": (
            "Rank members by predicted utilization risk so care-management "
            "resources (nurse outreach, care coordinators) reach High-Risk "
            "members first."
        ),
        "intended_uses": (
            "ML engineers and data scientists in the AAI-540 lab pipeline. "
            "Out of scope: not a diagnostic device; scores prioritize outreach "
            "only - never coverage or pricing decisions; synthetic data means "
            "demonstrated performance does NOT transfer to real members."
        ),
        "risk_rating": "Medium",
        "explanations_for_risk_rating": (
            "Allocation ethics: missing a High-Risk member has opportunity "
            "cost; mitigated by the High-Risk recall floor, human review, and "
            "PendingManualApproval gating production use."
        ),
    },
    "training_details": {
        "objective_function": {
            "function": "Minimize",
            "facet": "multiclass log loss (mlogloss)",
            "condition": "on the temporal validation slice (course 40/10/10/40 split)",
        },
        "training_observations": (
            "Single ml.m5.2xlarge training instance; ~33k train rows x "
            f"{champ_metrics['n_features']} one-hot+count features; XGBoost "
            "early-stopped on the validation slice; fixed seed 42."
        ),
        "training_job_details": {
            "training_arn": f"see SageMaker training job {RUN}",
            "training_datasets": [
                f"s3://{bucket}/{prefix}/processed/training/training_table.csv"
            ],
            "hyper_parameters": [
                {"name": k, "value": str(v)}
                for k, v in champ_metrics["hyperparameters"].items()
            ],
        },
    },
    "evaluation_details": [{
        "name": "holdout-test",
        "evaluation_observation": (
            f"Test macro F1 {champion['test_macro_f1']:.4f}; High-Risk recall "
            f"{champion['test_high_recall']:.4f}; OvR ROC-AUC "
            f"{champion['test_roc_auc']:.4f}; benchmark B2 macro F1 "
            f"{benchmark_test_f1:.4f}. Fairness slices recorded in this notebook."
        ),
        "evaluation_job_arn": f"training job {RUN}",
    }],
    "additional_information": {
        "ethical_considerations": (
            "Synthetic public-use data (CMS DE-SynPUF); no PHI/PII. Race is "
            "excluded from features and used only as an audit slice; per-slice "
            "recall gaps are reported above."
        ),
        "caveats_and_recommendations": (
            "Labels are rule-based terciles of future utilization (relative "
            "ranking, not clinical truth). Before any real-data use: external "
            "validation, recalibration, subgroup audits, and a CMS Data Use "
            "Agreement."
        ),
        "custom_details": {
            "label_rule": ("outcome_score_v1: total_visits + 3*inpatient + "
                           "drug_eras over 2009-2010, rank terciles"),
            "split_scheme": "course 40/10/10/40 train/validation/test/production",
            "champion_run": str(RUN),
        },
    },
}
sm_client.create_model_card(
    ModelCardName=f"{GROUP}-card",
    ModelCardStatus="Draft",
    Content=json.dumps(model_card),
)
print(f"created model card {GROUP}-card")

## 6. 📸 Verification describes (screenshot these three outputs)

In [ ]:
sm_client.describe_model_package_group(ModelPackageGroupName=GROUP)

In [ ]:
sm_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)

In [ ]:
sm_client.describe_model_card(ModelCardName=f"{GROUP}-card")